# hmslib — quicklook and detection

Workflow template: from a folder of CSVs to a calibrated Mahalanobis detector
per operating point.

Set `DATA_FOLDER` below to the folder holding the new data. Leaving it as
`None` generates a synthetic dataset, so the notebook runs end to end even
before any real data are available.

In [ ]:
import os
import sys

# the hmslib folder must be reachable: either next to this notebook,
# or add its parent directory here
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import hmslib as hm

hm.apply_style()
hm.set_seed(0)
hm.check_env()

## 1. Point at the data

`scan_folder` proposes a pairing between nominal and failure files and writes a
manifest. **Open the manifest and check it** before going on: it is the single
source of truth from here, and it is meant to be edited by hand (operating
point names, which column is the label, which one is the intensity, sensors to
exclude).

In [ ]:
DATA_FOLDER = None          # <-- put the real folder here
MANIFEST = "manifest.json"

if DATA_FOLDER is None:
    DATA_FOLDER = os.path.join(os.path.expanduser("~"), "hmslib_demo_data")
    hm.synth.write_dataset(DATA_FOLDER, n_ops=2, n_sensors=14, n_nominal=3000,
                           n_classes=6, n_per_class=600, random_state=0)
    print("using synthetic data in", DATA_FOLDER)

manifest = hm.io.scan_folder(DATA_FOLDER, write=MANIFEST)

In [ ]:
# what ended up in the manifest (edit the file if anything is wrong, then re-run)
print(open(MANIFEST, encoding="utf-8").read())

## 2. Load

Loading resolves the column roles. Read the `notes` printed below: every
inference is stated explicitly, and anything ambiguous is flagged. To override,
set `columns.label` / `columns.intensity` / `columns.exclude` in the manifest
(use the string `"none"` to declare that a column does not exist).

In [ ]:
ds = hm.Dataset.from_manifest(MANIFEST)
op = ds[ds.operating_points[0]]
op

## 3. Data quality — before any model

The questions that decide whether a covariance based method can work at all:
missing values, constant sensors, duplicated or collinear sensors, effective
rank. On engine data the sensors are redundant by construction, so this step
routinely finds something.

In [ ]:
rep = op.quality("nominal")
rep.describe()

In [ ]:
rep.pairs_frame()

## 4. Quicklook report

One PDF per operating point: structure, quality, distributions, correlations,
eigenvalue spectrum, PCA, class counts and — when an intensity column exists —
the sensor trends against fault intensity.

In [ ]:
hm.quicklook(ds, out="reports/")

## 5. Mahalanobis detector

Defaults: redundant sensors dropped, Ledoit-Wolf covariance, Cholesky
factorisation (never an explicit inverse), threshold taken as the out-of-sample
empirical quantile at `alpha`. All three threshold rules are computed so they
can be compared.

In [ ]:
det = hm.Mahalanobis(cov="ledoit_wolf", threshold="empirical", alpha=1e-3)
det.fit(op.nominal[op.sensors])
det.describe()

In [ ]:
det.threshold_table()

In [ ]:
s_nom = det.score(op.nominal[op.sensors])
s_fail = det.score(op.failures[op.sensors])

fig = hm.viz.plot_score_distributions(
    {"nominal": s_nom, "failures": s_fail}, threshold=det.threshold_)
plt.show()

print("false positive rate on nominal : %.3f%%" % (100 * np.mean(s_nom > det.threshold_)))
print("detection rate on failures     : %.1f%%" % (100 * np.mean(s_fail > det.threshold_)))

### Why the pre-check matters

Compare three chains at the same target false positive rate. The naive one
keeps every sensor and inverts the raw covariance; on data with a duplicated
sensor its actual false alarm rate can be an order of magnitude above the
target, because the chi-square threshold assumes more degrees of freedom than
the data carry.

In [ ]:
variants = {
    "default (drop + LW + cholesky)": hm.Mahalanobis(threshold="chi2", alpha=1e-3),
    "naive (keep all + empirical)": hm.Mahalanobis(
        cov="empirical", drop_redundant_sensors=False, threshold="chi2", alpha=1e-3),
    "truncated (eigen, 99% var)": hm.Mahalanobis(
        cov="empirical", drop_redundant_sensors=False, inverse="eigen",
        var_explained=0.99, threshold="chi2", alpha=1e-3),
}

rows = []
for name, model in variants.items():
    model.fit(op.nominal[op.sensors])
    sn = model.score(op.nominal[op.sensors])
    sf = model.score(op.failures[op.sensors])
    rows.append({
        "chain": name,
        "sensors": len(model.features_used_),
        "components": model.diagnostics_["n_components"],
        "threshold": model.threshold_,
        "FPR %": 100 * np.mean(sn > model.threshold_),
        "detected %": 100 * np.mean(sf > model.threshold_),
    })
pd.DataFrame(rows).round(3)

In [ ]:
fig = hm.viz.plot_eigen_spectrum(op.nominal, op.sensors)
plt.show()

## 6. From detection to isolation

The squared distance decomposes exactly over sensors. Averaged over the runs of
one failure class, the decomposition names the sensors that carry the anomaly.

In [ ]:
if op.label is not None:
    cls = op.classes[0]
    sub = op.failures[op.failures[op.label].astype(str) == cls]
    contrib = det.contributions(sub[op.sensors], normalize=True)
    fig = hm.viz.plot_contributions(contrib, top=10,
                                    title="contribution shares — %s" % cls)
    plt.show()
    display(det.top_contributors(sub[op.sensors].iloc[:5], k=4))

## 7. Sensor trends against fault intensity

Values in units of the nominal sigma, so the panels are comparable across
sensors: the Monte Carlo scatter in the background, the median over quantile
bins of intensity as a line, the interquartile band as a ribbon.

In [ ]:
if op.intensity is not None:
    fig = hm.viz.trend_vs_intensity(op, sensors=op.sensors[:6],
                                    classes=op.classes[:5], units="sigma")
    plt.show()
else:
    print("no intensity column: set columns.intensity in the manifest")

## 8. One model per operating point

The operating point is known at test time, so each one gets its own detector,
fitted on its own nominal cloud. `ModelBank` keeps them together, routes calls
by name and can be saved to disk for reuse.

In [ ]:
bank = hm.ModelBank.fit(ds, hm.Mahalanobis(threshold="empirical", alpha=1e-3))
bank.describe()

In [ ]:
table = bank.score_dataset(ds)
table.groupby(["op", "set"])["flagged"].agg(["mean", "count"])

In [ ]:
bank.save("models/")
# later, in another session:
# bank = hm.ModelBank.load("models/")